<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Contar Palabras como Cartas de una Baraja 🃏
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 09
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/09%20-%20Casos%20de%20Estudio%20y%20Aplicaciones/Para%20Dummies/02_NLP_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 02 del Módulo 09 (Procesamiento de lenguaje natural)](../02_Procesamiento_de_Lenguaje_Natural.ipynb). Una computadora no entiende frases; entiende **números**. Una forma muy sencilla de pasar de frases a números es **contar palabras**, como si cada frase fuera **una mano de cartas** y las palabras, las cartas: no importa el orden en que te las repartieron, solo **cuántas de cada tipo** tienes. Con esas cuentas se puede **adivinar si una reseña es positiva o negativa**.

Al terminar podrás explicar:
1. Cómo una frase se convierte en una **fila de conteos** (la «bolsa de palabras»).
2. Por qué las palabras **raras pesan más** que las comunes (TF-IDF).
3. Cómo un clasificador **«vota»** con las palabras (Naive Bayes) y por qué **contar palabras no entiende el «no»**.

> 🔗 Si luego quieres ver las versiones **desde cero y verificadas** (TF-IDF, Naive Bayes, LSA, y los *Transformers* como contexto): [cuaderno estándar](../02_Procesamiento_de_Lenguaje_Natural.ipynb). Antes de esto: [la lupa que se desliza](01_Vision_Computador_Dummies.ipynb). Después: [amigos con gustos parecidos](03_Recomendacion_Dummies.ipynb).

---
## 1. Frases convertidas en conteos 🃏

Tenemos una pequeña colección de **24 reseñas de un restaurante** escritas a mano, la mitad de gente a la que **le gustó** y la mitad a la que **no**. Para el modelo, cada frase será una **fila de conteos**: una columna por cada palabra distinta y, en cada casilla, **cuántas veces aparece** esa palabra en la frase. Veamos con tres frases cortas cómo queda.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

# --- Una "baraja" de reseñas escritas a mano (1 = le gustó, 0 = no le gustó) ---
RESENAS = [
    ("la comida estuvo deliciosa", 1), ("excelente servicio y muy amables", 1), ("me encantó el lugar", 1), ("la pizza es buenísima", 1), ("todo fue perfecto", 1),
    ("el postre estaba riquísimo", 1), ("atención rápida y comida sabrosa", 1), ("volveré pronto, genial", 1), ("muy buen ambiente y buenos precios", 1), ("la sopa es excelente", 1),
    ("no está mal la comida", 1), ("nada mal el servicio", 1),
    ("la comida llegó fría", 0), ("pésimo servicio y muy lento", 0), ("no me gustó el lugar", 0), ("la pizza es horrible", 0), ("todo fue un desastre", 0),
    ("el postre estaba salado", 0), ("atención lenta y comida mala", 0), ("no vuelvo jamás, terrible", 0), ("muy mal ambiente y precios caros", 0), ("la sopa es pésima", 0),
    ("no es bueno el servicio", 0), ("no me pareció nada rico", 0),
]
textos, y = [t for t, _ in RESENAS], np.array([e for _, e in RESENAS])

# --- Paso 1: contar palabras (bolsa de palabras): cada frase se vuelve una fila de numeros ---
mini = ["la pizza estuvo rica", "la pizza estuvo horrible", "la comida estuvo rica rica"]
cv = CountVectorizer()
tabla = pd.DataFrame(cv.fit_transform(mini).toarray(), columns=cv.get_feature_names_out(), index=[f"frase {i + 1}" for i in range(3)])
display(tabla)
print("Cada frase es ahora una fila de conteos: el ORDEN de las palabras se pierde (la «bolsa» de palabras), solo importa cuántas veces aparece cada una.")
assert tabla.loc["frase 3", "rica"] == 2 and tabla.loc["frase 1", "horrible"] == 0

---
**Lección:** la frase 3 («la comida estuvo rica rica») tiene un **2** en la columna *rica*. Fíjate en lo que **se pierde**: «la pizza estuvo rica» y «rica estuvo la pizza» darían **la misma fila**. La bolsa de palabras ignora el **orden**. Más abajo veremos que eso tiene un precio.

---

## 2. Las palabras raras pesan más ⚖️

Palabras como «la» o «el» aparecen en **casi todas** las frases: decir que una reseña contiene «la» no te dice nada. En cambio, que contenga «horrible» **sí** dice mucho, porque casi nadie la usa. Para darle más importancia a lo **raro**, a cada palabra se le asigna un **peso** que **sube cuanto menos aparece**: se llama **TF-IDF** (en el cuaderno estándar se calcula **desde cero**). Es como en una reunión: quien habla poco y dice algo distinto llama más la atención que quien repite lo de siempre.

In [ ]:
# --- Paso 2: las palabras RARAS pesan mas (TF-IDF): «la» aparece en todas partes y no dice nada; «horrible» dice mucho ---
tf = TfidfVectorizer()
M = tf.fit_transform(textos)
idf = dict(zip(tf.get_feature_names_out(), tf.idf_))
doc_por_palabra = {w: sum(w in t.split() for t in textos) for w in ["la", "no", "pizza", "horrible", "excelente"]}
print(f"{'palabra':<10}{'en cuántas reseñas (de 24)':>28}{'peso (idf)':>14}")
for w, n in doc_por_palabra.items():
    print(f"{w:<10}{n:>28}{idf[w]:>14.2f}")
print("→ Cuanto MÁS RARA es una palabra, MÁS pesa. Las que están en todas partes («la», «el») casi no informan.")
assert idf["horrible"] > idf["la"]

---
**Lección:** «la» está en 7 de las 24 reseñas y pesa **2.14**; «horrible» está en **una** y pesa **3.53**. Las palabras **raras** pesan más. (El peso depende de cuántas reseñas tenga la colección; lo que importa es el **orden**: raro → pesa más.)

---

## 3. Un clasificador que «vota» con las palabras 🗳️

El clasificador más sencillo para texto, **Naive Bayes** (*«ingenuo»*, porque supone que cada palabra vota **por separado**), funciona así: mira **en cuántas reseñas positivas y en cuántas negativas** apareció cada palabra y le asigna un **voto**: «deliciosa» sale casi siempre en las positivas, así que vota por *«gustó»*; «horrible», por *«no gustó»*. Para clasificar una frase nueva, **suma los votos** de sus palabras. Es la manera en que un jurado numeroso decide sumando opiniones individuales, sin hablar entre ellos.

In [ ]:
# --- Paso 3: un clasificador por "votacion de palabras" (Naive Bayes): cada palabra vota por «gustó» o «no gustó» segun donde aparecio mas ---
cv = CountVectorizer()
X = cv.fit_transform(textos)
nb = MultinomialNB(alpha=1.0).fit(X, y)
palabras = cv.get_feature_names_out()
voto = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]                            # >0 vota «gustó»; <0 vota «no gustó»
orden = np.argsort(voto)
fig, ax = plt.subplots(figsize=(9, 4.6))
idx = np.r_[orden[:8], orden[-8:]]
ax.barh(palabras[idx], voto[idx], color=["#dc2626" if v < 0 else "#16a34a" for v in voto[idx]])
ax.set_xlabel("Voto de la palabra (← «no gustó» | «gustó» →)"); ax.set_title("Las palabras que más empujan hacia cada lado", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()

nuevas = ["la comida es deliciosa y el servicio excelente", "el servicio fue horrible y lento", "me encantó, volveré pronto"]
for frase, p in zip(nuevas, nb.predict_proba(cv.transform(nuevas))[:, 1]):
    print(f"«{frase}» → probabilidad de que le haya gustado: {p:.0%}")
assert nb.predict(cv.transform([nuevas[0]]))[0] == 1 and nb.predict(cv.transform([nuevas[1]]))[0] == 0

---
**Lección:** las palabras que más empujan son las que esperaríamos («deliciosa», «excelente» hacia *gustó*; «horrible», «lento» hacia *no gustó*), y el modelo da probabilidades razonables a frases nuevas: ≈ 91 % de que le haya gustado la primera y ≈ 23 % de la segunda. Con solo 24 reseñas **todo esto es una demostración**: con tan pocos ejemplos, las probabilidades no son fiables.

---

## 4. La trampa del «no» 🚫

Este método tiene un **talón de Aquiles**: como solo cuenta palabras, **no distingue «es bueno» de «no es bueno»**. Para comprobarlo, generamos frases del tipo *«la comida es buena»* / *«la comida **no** es buena»* con muchos temas y adjetivos, entrenamos con unos temas y **probamos con temas que el modelo no vio**. Comparamos cuatro formas de mirar el texto: **palabras sueltas**, palabras sueltas **borrando el «no»** (algo que hacen algunas listas de «palabras vacías»), **pares** de palabras y **tríos** de palabras.

In [ ]:
# --- Paso 4: la trampa del «no». Generamos frases «X es bueno» / «X NO es bueno» con muchos temas y adjetivos ---
TEMAS = ["la comida", "el servicio", "el postre", "la pizza", "la sopa", "el café", "el lugar", "la música", "el mesero", "los precios"]
BUENOS, MALOS = ["bueno", "rico", "excelente", "genial", "perfecto", "sabroso"], ["malo", "feo", "horrible", "pésimo", "lento", "caro"]


def frases_de(temas):
    datos = []
    for t in temas:
        for adj in BUENOS:
            datos += [(f"{t} es {adj}", 1), (f"{t} no es {adj}", 0)]            # negar un adjetivo bueno -> NO gusto
        for adj in MALOS:
            datos += [(f"{t} es {adj}", 0), (f"{t} no es {adj}", 1)]            # negar un adjetivo malo -> SI gusto
    return [d for d, _ in datos], np.array([e for _, e in datos])


ent_t, ent_y = frases_de(TEMAS[:6])                                              # se entrena con 6 temas...
pru_t, pru_y = frases_de(TEMAS[6:])                                              # ...y se prueba con temas que NO vio
con_no = np.array(["no " in f for f in pru_t])


def prueba(nombre, vectorizador, quitar_no=False):
    quitar = lambda ts: [" ".join(w for w in t.split() if not (quitar_no and w == "no")) for t in ts]
    v = vectorizador; Xe = v.fit_transform(quitar(ent_t))
    m = LogisticRegression(C=10, max_iter=2000).fit(Xe, ent_y)
    pred = m.predict(v.transform(quitar(pru_t)))
    r = ((pred == pru_y).mean(), (pred == pru_y)[~con_no].mean(), (pred == pru_y)[con_no].mean())
    print(f"{nombre:<46}{r[0]:>12.0%}{r[1]:>20.0%}{r[2]:>20.0%}")
    return r


print(f"{'Representación':<46}{'Todas las frases':>12}{'Sin «no»':>20}{'Con «no»':>20}")
r1 = prueba("Palabras sueltas", CountVectorizer())
r2 = prueba("Palabras sueltas, borrando «no»", CountVectorizer(), quitar_no=True)
r3 = prueba("Palabras sueltas + pares", CountVectorizer(ngram_range=(1, 2)))
r4 = prueba("Palabras sueltas + pares + tríos", CountVectorizer(ngram_range=(1, 3)))
print("\n→ Con palabras sueltas (o con pares) el modelo NO distingue «es bueno» de «no es bueno»: «bueno» aparece igual de veces en frases que gustan y que no, así que no aporta nada y el resultado es el AZAR (50 %).")
print("→ Solo cuando aparecen los TRÍOS («no es bueno») el modelo ve la negación completa. Una bolsa de palabras no entiende el orden: para eso existen los modelos que leen en secuencia (los «transformers» del cuaderno estándar).")
print("Ojo: son frases inventadas con una estructura muy regular; con texto real hay mil maneras de negar, y hoy se resuelve con modelos que leen el contexto.")
assert r1[2] < 0.65 and r4[2] > 0.9 and abs(r1[0] - r2[0]) < 0.3

---
**Lección:** con palabras sueltas, con pares o borrando el «no», el resultado es **el azar (50 %)**: «bueno» aparece **tantas veces en frases que gustan como en frases que no** (en «no es bueno»), así que no ayuda. Solo con **tríos** («no es bueno») el modelo ve la negación completa (100 %). La moraleja no es «usa siempre tríos» (con texto real hay mil formas de negar) sino: **una bolsa de palabras no entiende el orden**; por eso hoy se usan modelos que **leen la frase en secuencia** (los *transformers*, del cuaderno estándar), y por eso **nunca debes borrar el «no»** sin medir qué pasa.

---

## Resumen en una frase 🎯

> **Para que una computadora clasifique texto, primero cuenta palabras (como cartas de una baraja), da más peso a las raras y deja que cada palabra «vote»; funciona sorprendentemente bien, pero no entiende el orden, así que el «no» la confunde si no se la ayuda.**

### Lo que aprendiste hoy:
- ✅ **Bolsa de palabras:** cada frase es una fila de conteos; se pierde el orden.
- ✅ **TF-IDF:** las palabras **raras** pesan más que las comunes («horrible» pesa más que «la»).
- ✅ **Naive Bayes:** cada palabra vota por una clase según dónde apareció más; se suman los votos.
- ✅ **El «no» es una trampa:** con palabras sueltas, «es bueno» y «no es bueno» son lo mismo; borrar el «no» **no arregla nada** y mirar tríos de palabras sí, pero solo en casos muy regulares.

> 🎓 **Siguiente paso:** [el cuaderno estándar](../02_Procesamiento_de_Lenguaje_Natural.ipynb) hace todo **desde cero y verificado contra scikit-learn**, compara el análisis por diccionario con el supervisado y explica cómo piensan los modelos modernos; o sigue con [amigos con gustos parecidos](03_Recomendacion_Dummies.ipynb).

---
##### 🛠️ Práctica 1: Que el jurado opine sobre tus propias frases

Ya tenemos el jurado entrenado: `cv` (la bolsa de palabras) y `nb` (Naive Bayes, que aprendió con las 24 reseñas). Ahora **escribe tú las frases**:

1. Crea una lista `mias` con tres frases: `"la sopa estuvo deliciosa"`, `"atención lenta y comida mala"` y `"excelente ambiente"`.
2. Conviértelas en conteos con `cv.transform(mias)` y pide a `nb` la probabilidad de que le haya **gustado** (`predict_proba`, columna 1). Guárdala en `prob_gusto`.
3. Imprime cada frase con su probabilidad en porcentaje.
4. Comprueba que la primera y la tercera salen con más de 50 % de gustar y la segunda con menos de 50 %.

In [ ]:
# Escribe tu código aquí

# mias = ["la sopa estuvo deliciosa", "atención lenta y comida mala", "excelente ambiente"]
# conteos = cv.transform(mias)
# prob_gusto = nb.predict_proba(conteos)[:, 1]
# for frase, pr in zip(mias, prob_gusto):
#     print(f"«{frase}» -> {pr:.0%}")


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
mias = ["la sopa estuvo deliciosa", "atención lenta y comida mala", "excelente ambiente"]
conteos = cv.transform(mias)                                   # cada frase -> fila de conteos (las palabras que el jurado no conoce no votan)
prob_gusto = nb.predict_proba(conteos)[:, 1]                   # columna 1 = probabilidad de «gustó»
for frase, pr in zip(mias, prob_gusto):
    print(f"«{frase}» -> probabilidad de que le haya gustado: {pr:.0%}")

assert prob_gusto[0] > 0.5 and prob_gusto[2] > 0.5 and prob_gusto[1] < 0.5
print("\nEl jurado suma los votos de las palabras que conoce: las positivas empujan hacia «gustó» y las negativas hacia «no gustó».")
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿qué se **gana** y qué se **pierde** al convertir una frase en una «bolsa de palabras»? Da un ejemplo de dos frases con **significados opuestos** que tendrían **la misma bolsa**.
2. ¿Por qué «la» pesa menos que «horrible» en TF-IDF? Si el texto fuera de **medicina**, ¿qué palabras pesarían poco y cuáles mucho?
3. Alguien propone «limpiar» las reseñas quitando todas las palabras cortas, incluido el «no». ¿Qué le responderías y qué comprobación harías antes de aceptar?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>